# 05 — RMST Concept Demonstration

The original project used year-specific adjusted RMST analyses at 36 and 60 months. This public notebook intentionally demonstrates the unadjusted RMST concept only, using synthetic data, and does not reproduce the original adjusted RMST analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..")
df = pd.read_csv(ROOT / "data" / "synthetic_dlbcl_demo.csv")
df.head()


In [ ]:
from statsmodels.duration.survfunc import SurvfuncRight

def km_rmst(times, events, tau):
    sf = SurvfuncRight(np.asarray(times), np.asarray(events))
    t = np.concatenate(([0.0], sf.surv_times))
    s = np.concatenate(([1.0], sf.surv_prob))
    area = 0.0
    for i, start in enumerate(t):
        if start >= tau:
            break
        end = t[i+1] if i+1 < len(t) else tau
        end = min(end, tau)
        if end > start:
            area += s[i] * (end-start)
    return area

for tau in [36,60]:
    print(f"tau = {tau} months")
    for group, g in df.groupby("treatment"):
        value = km_rmst(g["survival_months"], g["death"], tau)
        print(group, round(value, 2))
